# Component 4 — Garbage model v2 training (live-camera accuracy)
**R26-IT-143 · AI Smart Swimming Pool Monitoring · Garbage Intrusion Detection**

Run the cells **top to bottom**. Before you start:
1. `Runtime → Change runtime type → T4 GPU → Save`.
2. Your original Roboflow dataset is in Google Drive at `MyDrive/Dataset` (the folder with `data.yaml`, `train`, `valid`, `test`), as in your current notebook.
3. *(Strongly recommended)* Your OWN live-camera frames, labelled in Roboflow with the **same 6 class names**, exported as **YOLOv8** zip and uploaded to `MyDrive/my_pool_frames.zip`. Capture them with `tools/capture_dataset.py`. Include 20–30% empty-pool frames with no boxes.

What changed versus the first training: bigger model (**yolov8s**), bigger image size (**960**), more epochs with early stopping, live-camera style augmentation, your own camera frames, background (empty) frames, leakage check, and an honest test on your own camera.

## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - change runtime to T4!")

## 2. Mount Google Drive and install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install ultralytics

## 3. Settings — edit only this cell

In [ ]:
DRIVE = "/content/drive/MyDrive"
ORIG_DATASET = f"{DRIVE}/Dataset"                 # your Roboflow dataset (has data.yaml)
MY_FRAMES_ZIP = f"{DRIVE}/my_pool_frames.zip"     # your own labelled live frames (optional)
RUNS_DIR = f"{DRIVE}/c4_runs"                     # results are saved to Drive (survive disconnects)

MODEL = "yolov8s.pt"     # 's' is ~3x stronger than 'n' and still real-time on a laptop GPU.
                         # On a CPU-only laptop use "yolov8n.pt" and IMGSZ = 640.
IMGSZ = 960              # small/far floating objects are found much better than at 640
EPOCHS = 60              # ~3 min/epoch on a T4 at 960 px -> about 3 hours (fits a free session)
PATIENCE = 60            # = no early stop, so the learning rate decays fully and the final
                         #   no-mosaic epochs (close_mosaic) actually run - that is where
                         #   box accuracy (mAP50-95) improves most
CLASSES = ['aluminium foil', 'ball', 'bottle', 'juice', 'leaf', 'thermocol']

# Extra training images made by code from the data you already have (cell 5b).
USE_SYNTHETIC = True
SYNTH_N = 1500           # images with pasted objects (new places, sizes, angles)
SYNTH_NEGATIVES = 300    # empty-pool images (objects painted out) -> fewer false boxes on water
SYNTH_SCRIPT = f"{DRIVE}/synth_pool_data.py"   # upload tools/synth_pool_data.py here

## 4. Copy data to the fast local disk and merge with your own frames

In [ ]:
import os, shutil, glob, zipfile, yaml, random
from pathlib import Path

WORK = Path("/content/merged")
if WORK.exists(): shutil.rmtree(WORK)
for s in ["train", "valid", "test"]:
    (WORK / s / "images").mkdir(parents=True); (WORK / s / "labels").mkdir(parents=True)

def copy_split(src_root, src_split, dst_split, names_in, prefix):
    """Copies images+labels, re-mapping class ids BY NAME so a different class order can't break labels."""
    remap = {i: CLASSES.index(n) for i, n in enumerate(names_in) if n in CLASSES}
    img_dir = Path(src_root) / src_split / "images"
    n = 0
    for img in glob.glob(str(img_dir / "*")):
        if not img.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")): continue
        stem = Path(img).stem
        lab = Path(src_root) / src_split / "labels" / f"{stem}.txt"
        lines = []
        if lab.exists():
            for line in lab.read_text().splitlines():
                p = line.split()
                if p and int(p[0]) in remap:
                    lines.append(" ".join([str(remap[int(p[0])])] + p[1:]))
        shutil.copy(img, WORK / dst_split / "images" / f"{prefix}{Path(img).name}")
        (WORK / dst_split / "labels" / f"{prefix}{stem}.txt").write_text("\n".join(lines))  # empty file = background image
        n += 1
    return n

orig_names = yaml.safe_load(open(f"{ORIG_DATASET}/data.yaml"))["names"]
for s in ["train", "valid", "test"]:
    print("roboflow", s, copy_split(ORIG_DATASET, s, s, orig_names, "rf_"))

if os.path.exists(MY_FRAMES_ZIP):
    shutil.rmtree("/content/my_frames", ignore_errors=True)
    zipfile.ZipFile(MY_FRAMES_ZIP).extractall("/content/my_frames")
    my_yaml = glob.glob("/content/my_frames/**/data.yaml", recursive=True)[0]
    my_root = str(Path(my_yaml).parent)
    my_names = yaml.safe_load(open(my_yaml))["names"]
    for s in ["train", "valid", "test"]:
        if (Path(my_root) / s / "images").exists():
            print("my camera", s, copy_split(my_root, s, s, my_names, "my_"))
else:
    print("No own-camera frames found - training on Roboflow only (accuracy on YOUR camera will stay limited).")

## 5. Leakage check (important for honest numbers)
Roboflow names augmented copies `XXXX_jpg.rf.<hash>.jpg`. If copies of the **same original photo** are in both train and valid/test, the validation score is inflated (the model has already "seen" that photo). This cell removes those copies from **train**, so valid and test stay unchanged and the scores stay honest.

In [ ]:
def origin(name):  # "1013_jpg.rf.abc.jpg" -> "rf_1013"
    base = Path(name).stem
    return base.split("_jpg.rf.")[0].split(".rf.")[0]

held_out = {origin(p) for s in ["valid", "test"] for p in os.listdir(WORK/s/"images")}
removed = 0
for img in list(os.listdir(WORK/"train/images")):
    if origin(img) in held_out:
        os.remove(WORK/"train/images"/img)
        lab = WORK/"train/labels"/f"{Path(img).stem}.txt"
        if lab.exists(): os.remove(lab)
        removed += 1
print("leaked copies removed from train:", removed)
for s in ["train", "valid", "test"]:
    imgs = os.listdir(WORK/s/"images")
    empty = sum(1 for p in os.listdir(WORK/s/"labels") if os.path.getsize(WORK/s/"labels"/p) == 0)
    mine = sum(1 for p in imgs if p.startswith("my_"))
    print(f"{s:6s} images={len(imgs):5d}  own-camera={mine:4d}  background(empty)={empty}")

## 5b. Synthetic images (no new photos needed)
Pastes objects from your training images onto other pool images at new positions, sizes (incl. small/far), angles and lighting, partly "under water", and writes the labels itself. Also makes empty-pool images by painting the objects out. Only **train** gets synthetic images, so valid/test scores stay honest. Upload `tools/synth_pool_data.py` from the component zip to `MyDrive` first.

In [ ]:
import subprocess, sys
if USE_SYNTHETIC and os.path.exists(SYNTH_SCRIPT):
    yaml.safe_dump({"names": CLASSES, "nc": len(CLASSES)}, open(WORK / "data.yaml", "w"))
    subprocess.run([sys.executable, SYNTH_SCRIPT, "--data", str(WORK), "--out", "/content/synth",
                    "--n", str(SYNTH_N), "--negatives", str(SYNTH_NEGATIVES)], check=True)
    for sub in ["images", "labels"]:
        for f in os.listdir(f"/content/synth/{sub}"):
            shutil.move(f"/content/synth/{sub}/{f}", WORK / "train" / sub / f)
    print("train images now:", len(os.listdir(WORK / "train/images")))
else:
    print("Synthetic step skipped (USE_SYNTHETIC is False or synth_pool_data.py is not in MyDrive).")

## 6. Write the merged data.yaml

In [ ]:
data_yaml = WORK / "data.yaml"
yaml.safe_dump({"path": str(WORK), "train": "train/images", "val": "valid/images",
                "test": "test/images", "nc": len(CLASSES), "names": CLASSES},
               open(data_yaml, "w"), sort_keys=False)
print(open(data_yaml).read())

## 7. Train
Results and weights go to `MyDrive/c4_runs/garbage_v2`. If Colab disconnects, run the **Resume** cell below instead of training again.

In [ ]:
from ultralytics import YOLO
model = YOLO(MODEL)
model.train(
    data=str(data_yaml),
    epochs=EPOCHS, patience=PATIENCE,
    imgsz=IMGSZ, batch=-1,          # -1 = auto batch size for the T4
    project=RUNS_DIR, name="garbage_v2", exist_ok=True,
    save_period=10,                 # checkpoint every 10 epochs
    cos_lr=True, close_mosaic=15,
    # live-camera style augmentation: glare, colour shift, tilt, blur-like scale changes
    hsv_h=0.02, hsv_s=0.7, hsv_v=0.5,
    degrees=8, translate=0.15, scale=0.6, shear=2.0, perspective=0.0005,
    fliplr=0.5, flipud=0.0, mosaic=1.0, mixup=0.1,
    cache="disk", workers=2, seed=0,
)

### 7b. Resume (only if Colab disconnected during training)
After a disconnect, `/content` is wiped but your checkpoints are safe in Drive. **Do not run cell 7 again** (it would start from epoch 1). Run cells 1, 2, 3, 4, 5, 5b and 6 (they rebuild `/content/merged`, which training reads), then this cell. It continues from the last finished epoch.

In [ ]:
from ultralytics import YOLO
last = f"{RUNS_DIR}/garbage_v2/weights/last.pt"
assert os.path.exists(last), "No last.pt in Drive - training never finished an epoch; run cell 7 instead."
assert os.path.exists(WORK / "data.yaml"), "Run cells 4, 5, 5b and 6 first (they rebuild /content/merged)."
YOLO(last).train(resume=True)

### 7c. Fine-tune finish (use if training stopped early)
If training ended long before its last epoch (early stopping), the learning rate never came down and the final "no mosaic" epochs never ran. This continues from `best.pt` for 25 short epochs with a small learning rate and mosaic off for the last 10. About 75 minutes on a T4. Run cells 1-6 first if Colab was disconnected.

In [ ]:
from ultralytics import YOLO
start = f"{RUNS_DIR}/garbage_v2/weights/best.pt"
YOLO(start).train(
    data=str(data_yaml), epochs=25, patience=25, imgsz=IMGSZ, batch=-1,
    lr0=0.001, lrf=0.1, warmup_epochs=0, cos_lr=True, close_mosaic=10,
    project=RUNS_DIR, name="garbage_v2_ft", exist_ok=True, save_period=5,
    hsv_h=0.02, hsv_s=0.7, hsv_v=0.5, degrees=8, translate=0.15, scale=0.6,
    fliplr=0.5, mixup=0.0, cache="disk", workers=2, seed=0,
)
# Cells 8-11 use garbage_v2. To evaluate/export the fine-tuned model instead, run:
# best = f"{RUNS_DIR}/garbage_v2_ft/weights/best.pt"

## 8. Evaluate — valid split and YOUR camera test split

In [ ]:
from IPython.display import Image, display
best = f"{RUNS_DIR}/garbage_v2/weights/best.pt"
m = YOLO(best)
val = m.val(data=str(data_yaml), split="val", imgsz=IMGSZ)
test = m.val(data=str(data_yaml), split="test", imgsz=IMGSZ, name="test_eval")
for name, r in [("valid", val), ("test", test)]:
    print(f"{name}: P={r.box.mp:.3f} R={r.box.mr:.3f} mAP50={r.box.map50:.3f} mAP50-95={r.box.map:.3f}")
display(Image(filename=f"{RUNS_DIR}/garbage_v2/results.png", width=900))
display(Image(filename=f"{RUNS_DIR}/garbage_v2/confusion_matrix_normalized.png", width=700))

## 9. Compare OLD vs NEW model on the same test set (table for your thesis)
Upload your old `swimming_pool_garbage_yolo.pt` to `MyDrive` first.

In [ ]:
OLD = f"{DRIVE}/swimming_pool_garbage_yolo.pt"
if os.path.exists(OLD):
    old = YOLO(OLD).val(data=str(data_yaml), split="test", imgsz=640, name="old_test")
    print("OLD:", f"P={old.box.mp:.3f} R={old.box.mr:.3f} mAP50={old.box.map50:.3f} mAP50-95={old.box.map:.3f}")
    print("NEW:", f"P={test.box.mp:.3f} R={test.box.mr:.3f} mAP50={test.box.map50:.3f} mAP50-95={test.box.map:.3f}")

## 10. Test on a recorded LIVE clip (not a still image)
Record 20–30 s with Iriun (for example with `standalone_demo.py --record clip.mp4`), upload it to `MyDrive/test_clip.mp4`.

In [ ]:
CLIP = f"{DRIVE}/test_clip.mp4"
if os.path.exists(CLIP):
    m.predict(source=CLIP, conf=0.35, imgsz=IMGSZ, save=True, project="/content/pred", name="clip", exist_ok=True)
    print("Annotated video saved in /content/pred/clip/ - download it from the Files panel.")

## 11. Save the new model to Drive (then copy it into the project `models/` folder)

In [ ]:
shutil.copy(best, f"{DRIVE}/swimming_pool_garbage_yolo_v2.pt")
print("Saved:", f"{DRIVE}/swimming_pool_garbage_yolo_v2.pt")
from google.colab import files
files.download(f"{DRIVE}/swimming_pool_garbage_yolo_v2.pt")